# 06 · Implantação

**CRISP-DM fase 6 — implantação** · Telco Customer Churn

---

Atribui um segmento a clientes novos (ou à carteira inteira, em lote) em três passos:
1. **modelo:** lê o modelo vencedor do [`04-0-modelagem-cluster`](04-0-modelagem-cluster.ipynb) e a decisão do [`05-avaliacao`](05-avaliacao.ipynb);
2. **entrada:** o cliente em unidades de negócio (meses de casa, mensalidade, contrato…) passa **exatamente** pelo mesmo pré-processamento do [`03-preparacao`](03-preparacao.ipynb): a função `derivar_variaveis` e o `preprocessador.joblib` já ajustado;
3. **saída:** o segmento do cliente, com o perfil desse segmento na carteira e a ação de retenção sugerida.

No fim, um **monitoramento** simples compara um lote novo com a carteira de treino e diz quando re-treinar.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')  # silencia avisos de versão do scikit-learn

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.spatial.distance import cdist

DADOS = Path('../data/processed')
MODELOS = Path('../models')
RELATORIOS = Path('../reports')

## 1 · Modelo treinado e decisão da avaliação

In [ ]:
selecionado = json.loads((MODELOS / 'selecionado.json').read_text(encoding='utf-8'))      # vencedor e parâmetros, salvos pelo 04-0
avaliacao = json.loads((RELATORIOS / '05-avaliacao.json').read_text(encoding='utf-8'))     # decisão e nomes dos segmentos, salvos pelo 05
nome = {int(s): n for s, n in avaliacao['nomes'].items()} | {-1: 'Sem segmento'}           # mesmos nomes do 05

if not avaliacao['aprovado']:
    display(Markdown(f"> ⚠ **O 05 não aprovou a implantação** (próxima ação: *{avaliacao['proxima_acao']}*). "
                     'Este notebook roda como demonstração; os segmentos não devem ir para a operação.'))

try:
    modelo = joblib.load(MODELOS / selecionado['arquivo'])               # modelo treinado
except Exception as erro:                                                 # k-medianas/k-medoides só carregam onde a classe foi definida
    modelo = None
    print(f'Modelo não carregado ({type(erro).__name__}): a atribuição usa o centroide mais próximo')

if hasattr(modelo, 'predict'):
    metodo = 'predict do modelo'                                          # k-means, mini batch, gmm, birch…
elif hasattr(modelo, 'components_') and hasattr(modelo, 'eps'):
    metodo = 'núcleo mais próximo (dbscan)'                               # dbscan: vizinho núcleo a até eps, senão ruído
else:
    metodo = 'centroide mais próximo'                                     # ward e modelos que não carregaram

preprocessador = joblib.load(MODELOS / 'preprocessador.joblib')          # MinMax + one-hot, ajustado no 03
display(Markdown(
    f"Modelo: **{selecionado['modelo']}** · `{selecionado['params']}` · {selecionado['grupos']} segmentos · "
    f"atribuição por **{metodo}**"
))

## 2 · Pré-processamento do 03

O cliente novo passa primeiro pela `derivar_variaveis` (cópia fiel da função do 03: cria `ServicosProtecao` e junta os dois pagamentos automáticos) e depois pelo pré-processador salvo. A célula confere que a carteira inteira, passando por esse caminho, reproduz a saída do 03 antes de seguir.

In [ ]:
servicos_protecao = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']


def derivar_variaveis(dados):
    """Cria as variáveis derivadas da seção 1.3 do 03 a partir da base curada do notebook 02."""
    dados = dados.copy()

    # a) Quantos dos quatro serviços de proteção o cliente contrata (0 a 4)
    dados['ServicosProtecao'] = dados[servicos_protecao].eq('Yes').sum(axis=1)
    dados = dados.drop(columns=servicos_protecao)

    # b) Débito e cartão automáticos viram uma só categoria
    automaticos = {'Bank transfer (automatic)': 'Automatic',
                   'Credit card (automatic)': 'Automatic'}
    dados['PaymentMethod'] = dados['PaymentMethod'].replace(automaticos)
    return dados


curada = pd.read_csv(DADOS / 'Telcon-Customer-Churn_tratado.csv', sep=',')                 # base curada: referência de categorias e faixas
X_base = preprocessador.transform(derivar_variaveis(curada))                                 # carteira no espaço do modelo
saida_03 = pd.read_csv(DADOS / 'Telcon-Customer-Churn_preprocessed.csv', sep=',')
assert list(X_base.columns) == list(saida_03.columns) and np.allclose(X_base, saida_03), 'o pré-processamento não reproduz o 03'
print(f'Pré-processamento confere com o 03: {len(X_base)} clientes × {X_base.shape[1]} features')

In [ ]:
segmentos = pd.read_csv(DADOS / 'segmentos.csv', sep=',')['segmento'].to_numpy()          # rótulos da carteira (04-0)
validos = segmentos >= 0
centroides = X_base[validos].groupby(segmentos[validos]).mean()                              # centro de cada segmento

if metodo == 'núcleo mais próximo (dbscan)':
    nucleos = modelo.components_                                                             # clientes núcleo do dbscan
    rotulo_nucleo = modelo.labels_[modelo.core_sample_indices_]


def atribuir(X_novo):
    # Segmento de cada linha de X_novo (já no espaço do modelo), pelo método disponível.
    if metodo == 'predict do modelo':
        return modelo.predict(X_novo)
    if metodo == 'núcleo mais próximo (dbscan)':
        dist = cdist(X_novo, nucleos)
        return np.where(dist.min(axis=1) <= modelo.eps, rotulo_nucleo[dist.argmin(axis=1)], -1)  # longe de todo núcleo = ruído
    return centroides.index[cdist(X_novo, centroides).argmin(axis=1)].to_numpy()


concordancia = (atribuir(X_base.to_numpy()) == segmentos).mean()                            # a atribuição reproduz a carteira?
print(f'Concordância da atribuição com segmentos.csv: {concordancia:.1%}')

perfil = pd.read_csv(RELATORIOS / '05-perfil-segmentos.csv', sep=';', index_col='segmento')  # perfil de cada segmento (05)
perfil.style.format({'tenure': '{:.0f}', 'mensalidade': '${:,.2f}', 'gasto_total': '${:,.0f}', 'receita_mensal': '${:,.0f}',
                     'protecao': '{:.1f}', 'contrato_mensal': '{:.0%}', 'fibra': '{:.0%}', 'cheque_eletronico': '{:.0%}',
                     'idoso': '{:.0%}', 'participacao': '{:.1%}'})

## 3 · Ação sugerida por segmento

A ação sai do perfil do segmento, por uma regra simples e explícita. **É uma proposta para a equipe de retenção validar**, não uma saída do modelo:
* **prioridade** pelo risco relativo à carteira: a fatia do segmento em contrato mês a mês ÷ a da carteira (o cliente mensal pode sair a qualquer momento). Acima de 1,2× é alta, abaixo de 0,8× é baixa;
* **oferta** pelo traço mais marcante do perfil.

In [ ]:
segmentados = perfil.drop(index='Sem segmento', errors='ignore')
mensal_carteira = (perfil['contrato_mensal'] * perfil['clientes']).sum() / perfil['clientes'].sum()
risco = segmentados['contrato_mensal'] / mensal_carteira                                     # contrato mensal do segmento ÷ o da carteira


def acao(p, r):
    if p['contrato_mensal'] >= 0.6 and p['tenure'] <= 12:
        oferta = 'onboarding nos primeiros meses e desconto para migrar a contrato anual'
    elif p['fibra'] >= 0.6 and p['protecao'] < 1.5:
        oferta = 'pacote de proteção/suporte técnico sem custo por 3 meses (fibra com pouca proteção)'
    elif p['contrato_mensal'] >= 0.5:
        oferta = 'migração para contrato anual com desconto na mensalidade'
    else:
        oferta = 'programa de fidelidade e renovação antecipada do contrato'
    prioridade = 'alta' if r >= 1.2 else 'baixa' if r <= 0.8 else 'média'
    return pd.Series({'prioridade': prioridade, 'ação sugerida': oferta})


playbook = segmentados.apply(lambda p: acao(p, risco[p.name]), axis=1).assign(risco_relativo=risco)
playbook.loc['Sem segmento'] = ['revisar', 'cliente fora dos padrões da carteira: análise individual', np.nan]
playbook.style.format({'risco_relativo': '{:.2f}×'}, na_rep='—')

## 4 · Função de atribuição

Cada cliente é um dicionário com as colunas da base curada (as mesmas do 02). **Todas as colunas que o modelo usa são obrigatórias**; `gender` e `TotalCharges` são opcionais, porque o 03 as deixou fora.

Coluna faltando, categoria inexistente ou valor fora do tipo geram erro com as opções válidas, em vez de um segmento errado silencioso. Valores numéricos fora da faixa da carteira são aceitos, mas geram aviso: o MinMax os leva para fora de 0–1.

In [ ]:
opcionais = {'gender', 'TotalCharges', 'customerID'}                                       # o 03 deixa fora; customerID só identifica
obrigatorias = [c for c in curada.columns if c not in opcionais]
numericas = [c for c in obrigatorias if pd.api.types.is_numeric_dtype(curada[c])]
categorias = {c: sorted(curada[c].unique()) for c in obrigatorias if c not in numericas}
faixa = curada[numericas].agg(['min', 'max'])


def validar(clientes):
    # Confere colunas, categorias e tipos; devolve a lista de avisos (valores fora da faixa da carteira).
    avisos = []
    for i, cliente in enumerate(clientes, start=1):
        faltando = [c for c in obrigatorias if c not in cliente]
        if faltando:
            raise ValueError(f'Cliente {i}: faltam as colunas {faltando}')
        sobrando = set(cliente) - set(obrigatorias) - opcionais
        if sobrando:
            raise ValueError(f'Cliente {i}: colunas que não existem na base curada: {sorted(sobrando)}')
        for col, validos_col in categorias.items():
            if cliente[col] not in validos_col:
                raise ValueError(f'Cliente {i}, {col}: "{cliente[col]}" não existe na base. Válidos: {validos_col}')
        for col in numericas:
            try:
                valor = float(cliente[col])
            except (TypeError, ValueError):
                raise ValueError(f'Cliente {i}, {col}: "{cliente[col]}" não é numérico') from None
            if not faixa.loc['min', col] <= valor <= faixa.loc['max', col]:
                avisos.append(f'Cliente {i}, {col} = {valor:g} fora da faixa da carteira '
                              f'({faixa.loc["min", col]:g} a {faixa.loc["max", col]:g})')
    return avisos


def segmentar(clientes):
    for aviso in validar(clientes):
        print('Aviso:', aviso)
    entrada = pd.DataFrame(clientes)
    for col in opcionais - {'customerID'}:                                                    # opcionais ausentes: valor neutro, não entram no modelo
        if col not in entrada:
            entrada[col] = curada[col].mode()[0]
    entrada[numericas] = entrada[numericas].astype(float)

    X_novo = preprocessador.transform(derivar_variaveis(entrada[curada.columns])).to_numpy()  # mesmo espaço do modelo
    segmento = pd.Series(atribuir(X_novo)).map(nome)
    return entrada.assign(segmento=segmento.to_numpy(),
                          prioridade=segmento.map(playbook['prioridade']).to_numpy(),
                          acao=segmento.map(playbook['ação sugerida']).to_numpy())

## 5 · Entrada: novos clientes

Edite a lista abaixo e rode as duas células seguintes. Para um lote, use `segmentar(pd.read_csv('arquivo.csv', sep=',').to_dict('records'))`.

Os valores seguem a base curada do 02: `SeniorCitizen` é `Yes`/`No`, e `MultipleLines` e os serviços de internet são `No` para quem não tem telefone ou internet.

In [ ]:
novos_clientes = [
    {   # cliente recém-chegado, fibra, contrato mensal, boleto eletrônico, sem proteção
        'SeniorCitizen': 'No', 'Partner': 'No', 'Dependents': 'No', 'tenure': 2,
        'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No',
        'StreamingTV': 'Yes', 'StreamingMovies': 'No', 'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 85.0,
    },
    {   # cliente antigo, família, contrato de dois anos, débito automático, pacote completo
        'SeniorCitizen': 'No', 'Partner': 'Yes', 'Dependents': 'Yes', 'tenure': 60,
        'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'InternetService': 'DSL',
        'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes', 'DeviceProtection': 'Yes', 'TechSupport': 'Yes',
        'StreamingTV': 'Yes', 'StreamingMovies': 'Yes', 'Contract': 'Two year',
        'PaperlessBilling': 'No', 'PaymentMethod': 'Bank transfer (automatic)', 'MonthlyCharges': 89.0,
    },
    {   # só telefone, sem internet, contrato anual, boleto por correio
        'SeniorCitizen': 'No', 'Partner': 'Yes', 'Dependents': 'No', 'tenure': 24,
        'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'No',
        'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No',
        'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'One year',
        'PaperlessBilling': 'No', 'PaymentMethod': 'Mailed check', 'MonthlyCharges': 20.0,
    },
]

## 6 · Saída: segmento de cada cliente

In [ ]:
resultado = segmentar(novos_clientes)
display(resultado[['tenure', 'InternetService', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'segmento', 'prioridade', 'acao']])

for i, linha in resultado.iterrows():                                                      # perfil do segmento atribuído
    p = perfil.loc[linha['segmento']]
    display(Markdown(
        f"**Cliente {i + 1} → {linha['segmento']}** (prioridade {linha['prioridade']}) · {p['clientes']:.0f} clientes na carteira · "
        f"tenure mediano {p['tenure']:.0f} meses · mensalidade mediana ${p['mensalidade']:,.2f} · {p['contrato_mensal']:.0%} em contrato mensal  \n"
        f"Ação sugerida: {linha['acao']}"
    ))

In [ ]:
# Um cliente com erro de digitação é recusado com as opções válidas, em vez de cair num segmento qualquer
try:
    segmentar([{**novos_clientes[0], 'Contract': 'Mensal'}])
except ValueError as erro:
    print('Recusado:', erro)

## 7 · Lote: lista de ação para a equipe de retenção

A carteira inteira passa pela mesma função e vira uma lista `customerID` → segmento → prioridade → ação. Dentro de cada prioridade, vêm primeiro os clientes em contrato mês a mês e com menos tempo de casa, que podem sair a qualquer momento. É o arquivo que a equipe de retenção usa. A conferência garante que o caminho da implantação dá o mesmo segmento que o 04-0 deu a cada cliente.

In [ ]:
bruta = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv', sep=',',
                    usecols=['customerID', 'tenure'])                                        # só o customerID (mesma ordem do 02)
assert np.array_equal(bruta['tenure'], curada['tenure']), 'a base curada não está na ordem da base bruta'

carteira = segmentar(curada.assign(customerID=bruta['customerID']).to_dict('records'))
assert (carteira['segmento'].to_numpy() == pd.Series(segmentos).map(nome).to_numpy()).mean() == concordancia

ordem_prioridade = {'alta': 0, 'média': 1, 'baixa': 2, 'revisar': 3}
lista_acao = (carteira[['customerID', 'segmento', 'prioridade', 'acao', 'tenure', 'Contract', 'MonthlyCharges']]
              .assign(_p=lambda d: d['prioridade'].map(ordem_prioridade), _m=lambda d: d['Contract'].ne('Month-to-month'))
              .sort_values(['_p', '_m', 'tenure'])                               # prioridade → contrato mensal primeiro → menos tempo de casa
              .drop(columns=['_p', '_m']))
RELATORIOS.mkdir(exist_ok=True)
lista_acao.to_csv(RELATORIOS / '06-lista-acao-retencao.csv', sep=',', index=False)
print(f'{len(lista_acao):,} clientes gravados em {RELATORIOS / "06-lista-acao-retencao.csv"}'.replace(',', '.'))
display(lista_acao.groupby(['prioridade', 'segmento'], sort=False).size().rename('clientes').to_frame())
lista_acao.head(10)

## 8 · Monitoramento e manutenção

O modelo foi ajustado numa fotografia da carteira. Quando o perfil dos clientes muda (novos planos, reajuste de preço, outra política comercial), os segmentos envelhecem. Para cada lote novo, o monitoramento mede o **PSI** (*population stability index*) de cada variável contra a carteira de treino e a distribuição dos segmentos atribuídos.

| PSI | Leitura | Ação |
|---|---|---|
| < 0,10 | estável | nenhuma |
| 0,10 a 0,25 | mudança moderada | acompanhar no próximo lote |
| > 0,25 | mudança relevante | re-treinar (refazer 03 → 04 → 05) |

In [ ]:
def psi(referencia, atual, faixas=10):
    # Population stability index: soma de (a − r) · ln(a / r) sobre as faixas da referência.
    if pd.api.types.is_numeric_dtype(referencia):
        bordas = np.unique(np.quantile(referencia, np.linspace(0, 1, faixas + 1)))
        bordas[0], bordas[-1] = -np.inf, np.inf
        r = pd.cut(referencia, bordas).value_counts(normalize=True, sort=False)
        a = pd.cut(atual, bordas).value_counts(normalize=True, sort=False)
    else:
        r, a = referencia.value_counts(normalize=True), atual.value_counts(normalize=True)
        r, a = r.align(a, fill_value=0)
    r, a = r.clip(lower=1e-4), a.clip(lower=1e-4)                                           # evita log de zero
    return float(((a - r) * np.log(a / r)).sum())


def monitorar(lote):
    # Lote em unidades de negócio (mesmas colunas de novos_clientes) → PSI por variável e mix de segmentos.
    atribuido = segmentar(lote.to_dict('records'))
    variaveis = pd.Series({c: psi(curada[c], atribuido[c]) for c in obrigatorias}).sort_values(ascending=False)
    mix = pd.DataFrame({'carteira': pd.Series(segmentos).map(nome).value_counts(normalize=True),
                        'lote': atribuido['segmento'].value_counts(normalize=True)}).fillna(0)
    psi_mix = psi(pd.Series(segmentos).map(nome), atribuido['segmento'])
    pior = variaveis.max()
    leitura = 're-treinar' if max(pior, psi_mix) > 0.25 else 'acompanhar' if max(pior, psi_mix) > 0.10 else 'estável'
    return variaveis, mix, psi_mix, leitura


# Demonstração: um lote simulado só de clientes novos (tenure ≤ 6) desloca o perfil da carteira
lote_simulado = curada[curada['tenure'] <= 6].sample(500, random_state=42)
variaveis, mix, psi_mix, leitura = monitorar(lote_simulado)
display(Markdown(f'**Lote simulado (500 clientes com tenure ≤ 6)** · PSI do mix de segmentos: **{psi_mix:.2f}** · leitura: **{leitura}**'))
display(pd.concat([variaveis.head(6).rename('PSI').to_frame().T]).style.format('{:.2f}'))
mix.style.format('{:.1%}')

## Plano de implantação

| Item | Definição |
|---|---|
| **Entrega** | `reports/06-lista-acao-retencao.csv` (carteira) e a função `segmentar` (clientes novos), que pode ir para `src/deployment/app.py` (Streamlit) |
| **Frequência** | lote mensal da carteira ativa · clientes novos no cadastro |
| **Responsável pelo uso** | equipe de retenção: aplica a ação sugerida por prioridade e registra o resultado |
| **Medição do efeito** | grupo de controle aleatório em cada segmento (sem ação), para medir os cancelamentos evitados de verdade e validar as premissas do 05 |
| **Re-treino** | semestral, ou quando o monitoramento indicar PSI > 0,25 |
| **Artefatos versionados** | `models/preprocessador.joblib`, `models/<modelo>.joblib`, `models/selecionado.json`, `reports/05-avaliacao.json` |
| **Riscos** | ação baseada em segmento e não em risco individual · premissas financeiras não validadas · prioridade baseada em indicador (contrato mês a mês), sem rótulo de cancelamento |